# AAROH Escalation Assessment Model

This notebook trains the production interpretable logistic regression model using case-level splits and the existing synthetic demonstration supervision. Outputs are operational assessment signals only, not clinical ground truth or diagnoses.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/AAROH'
import os
if not os.path.isdir(REPO_DIR):
    raise FileNotFoundError(f'Expected the AAROH repository at {REPO_DIR}')
os.chdir(REPO_DIR)

In [ ]:
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/AAROH')
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints' / 'escalation'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
Path('models/escalation').mkdir(parents=True, exist_ok=True)
print('Drive checkpoint directory:', CHECKPOINT_DIR)

In [ ]:
!python -m backend.ml.training.train_escalation \
  --output-dir models/escalation \
  --checkpoint-dir checkpoints/escalation \
  --drive-checkpoint-dir $CHECKPOINT_DIR \
  --target-horizon-days 7 \
  --threshold-low-moderate 0.40 \
  --threshold-moderate-high 0.75 \
  --epochs 150 \
  --lr 0.08

In [ ]:
!python -m backend.ml.training.evaluate_escalation_model \
  --model-dir models/escalation \
  --output-file models/escalation/metrics.json \
  --case-count 40

In [ ]:
from pathlib import Path
required = [
    Path('models/escalation/config.json'),
    Path('models/escalation/metadata.json'),
    Path('models/escalation/weights'),
    Path('models/escalation/checkpoint'),
    Path('models/escalation/metrics.json'),
    Path('models/escalation/label_mapping.json'),
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing exported Escalation artifacts: {missing}')
print('Escalation artifacts exported successfully')